# Study 809 — Signed Jump Variation — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 4126, 'spread_bps': -1.71, 't_nw': -1.36, 't_1s': -1.29, 'lo_bps': 7.2, 'hi_bps': 8.91, 'welch_t': -0.66, 'gross_sharpe': -0.32, 'placebo_obs': -1.71, 'placebo_mean': 0.025, 'placebo_sd': 0.899, 'placebo_p': 0.974, 'placebo_sigma_left': 1.9, 'placebo_draws': 1000, 'era_early_bps': -0.35, 'era_early_t': -0.26, 'era_early_n': 1992, 'era_late_bps': -2.98, 'era_late_t': -1.44, 'era_late_n': 2134, 'timer_1_gross': -1.71, 'timer_1_cost': 2.14, 'timer_1_net': -3.85, 'timer_1_t': -2.89, 'timer_5_gross': -1.71, 'timer_5_cost': 10.14, 'timer_5_net': -11.85, 'timer_5_t': -8.91, 'null_mean_t': -0.31, 'null_sd_t': 0.82, 'null_fire': 0, 'planted_t': 3.66, 'planted_welch': 3.57}

## The headline — long-low-SJ / short-high-SJ spread

Daily equal-weight bottom-30% (low SJ) minus top-30% (high SJ) signed-jump spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : low-SJ {R['lo_bps']:+.2f} vs high-SJ {R['hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : -1.71 bps/day  NW(10) t = -1.36  one-sample t = -1.29
books         : low-SJ +7.20 vs high-SJ +8.91 bps (Welch t = -0.66)
gross Sharpe  : -0.32 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> right-tail p = {R['placebo_p']:.5f} "
      f"(~{R['placebo_sigma_left']:.1f}sigma into the LEFT tail)")

observed -1.71 bps vs placebo mean +0.025 (sd 0.899) -> right-tail p = 0.97400 (~1.9sigma into the LEFT tail)


## Robustness — two eras (split 2018-01-01)

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1992): -0.35 bps  NW t = -0.26
2018-2026 (n=2134): -2.98 bps  NW t = -1.44


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross -1.71 -> net -3.85 bps/day (cost 2.14/day, t=-2.89)
5 bps one-way: gross -1.71 -> net -11.85 bps/day (cost 10.14/day, t=-8.91)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from signed_jump import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=809+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0024, seed=809, n_assets=40, n_days=1500))
print(f"planted (edge=0.0024): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.53 (sd 0.62), |t|>=2 in 0/8


planted (edge=0.0024): NW t = +3.66, Welch t = +3.57


## Verdict

- **Signal — None.** The claimed Bollerslev-Li-Zhao negative-signed-jump premium does **not** replicate on 50 liquid US mega-caps: the long-low-SJ / short-high-SJ spread is **-1.71 bps/day** (NW *t* = **-1.36**, |*t*| < 2) — insignificant *and* mildly *wrong-signed* (the permutation null centres at 0, sd 0.90 bps; observed ~1.9σ into the left tail), and flat in both eras (*t* = -0.26 / -1.44). The 20-seed synthetic control recovers a *planted* relation cleanly (*t* = +3.66, fires on 0/20 nulls), so the absence is real, not machinery. Survivorship biases the magnitude.
- **Tradability — Mirage.** The specified book loses money net at any cost: at 1 bp one-way the friction (2.14 bps/day) already dwarfs the 1.71 bps (insignificant) gross edge, net **-3.85 bps/day** (*t* = -2.89); at 5 bps **-11.85 bps/day**.